# Notebook Overview — Run Representation-Based VideoQA

## Purpose

This notebook performs representation-based Video Question Answering (VideoQA) using precomputed video and text representations. Rather than performing direct multimodal inference, the notebook combines shared `clip_text` representations with either `clip_video` or `autoencoder_video` representations and trains a Fusion MLP classifier to predict the correct multiple-choice answer.

The notebook prepares the evaluation dataset, loads the required representation artifacts, constructs multimodal training examples, trains the shared Fusion MLP classifier, generates multiple-choice predictions, validates the resulting prediction dataset, and saves experiment artifacts for downstream evaluation.

The prediction pipeline is identical for both representation-based experiments. The only experimental difference is the source of the video representations.

## Inputs

* Shared `clip_text` representation artifacts
* `clip_video` or `autoencoder_video` representation artifacts
* NExT-QA evaluation annotations
* Shared project configuration

## Outputs

* Multiple-choice prediction dataset
* Prediction validation report
* Experiment summary report
* Saved prediction artifacts for Notebook 08

## Processing Workflow

1. Initialize the notebook environment.
2. Configure the representation-based VideoQA experiment.
3. Verify runtime readiness.
4. Prepare the NExT-QA evaluation dataset.
5. Load shared `clip_text` representations.
6. Load the selected video representations.
7. Prepare multimodal training examples.
8. Train the Fusion MLP classifier and generate multiple-choice predictions.
9. Validate prediction artifacts.
10. Save experiment artifacts.
11. Generate prediction summary reports.
12. Display representative predictions.
13. Summarize the completed experiment.

## Downstream Consumer

Notebook 08 — Evaluate Development Results


### 🔷 Step 1 — Initialize Environment for Representation-Based VideoQA

* Mount Google Drive and prepare the Colab execution environment.
* Clone the private project repository and move into the repository directory.
* Load shared project configuration constants and utility modules.
* Verify required project paths and initialize local output directories.
* Load NExT-QA annotation metadata for evaluation dataset preparation.
* Verify that the required shared CLIP text and video representation artifacts are available.
* Display dataset split summary information when verbose output is enabled.
* Prepare the notebook for representation-based VideoQA inference.


In [ ]:
# ============================================================
# Step 1: Initialize Environment for Representation-Based VideoQA
# ============================================================

VERBOSE = True

import os
from pathlib import Path
import shutil

import pandas as pd
from google.colab import drive, userdata

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# REPO SETUP
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):
    print("Cloning project repository...")
    !git clone --quiet --filter=blob:none --no-checkout {repo_url}
    os.chdir(REPO_DIR)
    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main
else:
    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD CONFIG
# ------------------------------------------------------------

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# EXPERIMENT SETUP
# ------------------------------------------------------------

EXPERIMENT_NAME = "ae_seg6s_stride4_dev25"
#EXPERIMENT_NAME = "clip_video_dev25"

configure_experiment(EXPERIMENT_NAME)

EXPERIMENT_TYPE = infer_experiment_type(EXPERIMENT_NAME)

print(f"Experiment name : {EXPERIMENT_NAME}")
print(f"Experiment type : {EXPERIMENT_TYPE}")

# ------------------------------------------------------------
# REQUIRED PATHS
# ------------------------------------------------------------

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    Path("src"),
    Path("datasets"),
    QUESTIONS_DIR,
    METADATA_DIR,
    OUTPUTS_DIR,
]

missing_paths = [p for p in required_paths if not p.exists()]

if missing_paths:
    raise FileNotFoundError(f"Missing paths: {missing_paths}")

# ------------------------------------------------------------
# LOAD NExT-QA
# ------------------------------------------------------------

print("\nLoading NExT-QA annotations...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(split_annotations, verbose=VERBOSE)

split_summary_df = summarize_nextqa_splits(annotations_df)

if annotations_df.empty:
    raise RuntimeError("No NExT-QA annotation records loaded.")

# ------------------------------------------------------------
# ARTIFACT RESTORE (STRICT RULE ENFORCEMENT)
# ------------------------------------------------------------

print("\nRestoring representation artifacts...")

# ------------------------------------------------------------
# ALWAYS COPY: CLIP TEXT
# ------------------------------------------------------------

artifact_restore_pairs = [
    {
        "name": "CLIP text representations",
        "drive_path": CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV,
        "local_path": CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV,
    }
]

# ------------------------------------------------------------
# CONDITIONAL VIDEO SOURCE (ONLY ONE)
# ------------------------------------------------------------

if EXPERIMENT_NAME.startswith("clip"):

    video_name = "CLIP video representations"
    video_drive = CLIP_VIDEO_REPRESENTATIONS_DRIVE_CSV
    video_local = CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV

elif EXPERIMENT_NAME.startswith("ae_"):

    video_name = "Autoencoder video representations"
    video_drive = AUTOENCODER_VIDEO_REPRESENTATIONS_CSV
    video_local = AUTOENCODER_LOCAL_VIDEO_REPRESENTATIONS_CSV

else:
    raise ValueError(
        f"Unknown experiment type for video selection: {EXPERIMENT_NAME}"
    )

artifact_restore_pairs.append({
    "name": video_name,
    "drive_path": video_drive,
    "local_path": video_local,
})

# ------------------------------------------------------------
# COPY ARTIFACTS
# ------------------------------------------------------------

for artifact in artifact_restore_pairs:

    drive_path = Path(artifact["drive_path"])
    local_path = Path(artifact["local_path"])

    if not drive_path.exists():
        raise FileNotFoundError(
            f"Missing Drive artifact: {artifact['name']} -> {drive_path}"
        )

    local_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        print(f"Copying {artifact['name']} to local runtime...")
        shutil.copy2(drive_path, local_path)
    else:
        print(f"Already exists locally: {local_path}")

print("\nArtifact restoration complete.")

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\nDataset loaded.")
print(f"Annotation records: {len(annotations_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment ready for VideoQA.")



### 🔷 Step 2 — Define Representation-Based VideoQA Configuration

* Configure development-subset or full-dataset execution for representation-based VideoQA.
* Select the active video representation source (`clip_video` or `autoencoder_video`).
* Specify the shared text representation source (`clip_text`).
* Configure the Fusion MLP classifier used for multiple-choice prediction.
* Define the evaluation split, random seed, answer mode, and experiment parameters.
* Configure representation input artifacts and prediction output locations.
* Display the active experiment configuration.


In [ ]:
# ============================================================
# Step 2: Define Representation-Based VideoQA Configuration
# ============================================================

print("Defining representation-based VideoQA configuration...")

# ------------------------------------------------------------
# Execution controls
# ------------------------------------------------------------

RUN_FULL_EVALUATION_SPLIT = False

# ------------------------------------------------------------
# Representation source configuration (CONFIG-DRIVEN)
# ------------------------------------------------------------

TEXT_REPRESENTATION_SOURCE = DEFAULT_TEXT_REPRESENTATION_SOURCE

# ------------------------------------------------------------
# REPRESENTATION SOURCE MUST BE DERIVED FROM EXPERIMENT
# ------------------------------------------------------------
if EXPERIMENT_TYPE == "autoencoder":
    VIDEO_REPRESENTATION_SOURCE = AUTOENCODER_VIDEO_REPRESENTATION_SOURCE

elif EXPERIMENT_TYPE == "clip_video":
    VIDEO_REPRESENTATION_SOURCE = CLIP_VIDEO_REPRESENTATION_SOURCE

else:
    raise ValueError(
        f"Unsupported EXPERIMENT_TYPE: {EXPERIMENT_TYPE}"
    )

# ------------------------------------------------------------
# Development / evaluation configuration
# ------------------------------------------------------------

evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

answer_mode = ANSWER_MODE
choice_columns = CHOICE_COLUMNS
question_column = QUESTION_COLUMN
video_id_column = VIDEO_ID_COLUMN
ground_truth_answer_column = GROUND_TRUTH_ANSWER_COLUMN

# ------------------------------------------------------------
# Input artifact configuration
# ------------------------------------------------------------

TEXT_REPRESENTATIONS_CSV = CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV

if VIDEO_REPRESENTATION_SOURCE == "clip_video":
    VIDEO_REPRESENTATIONS_CSV = CLIP_VIDEO_REPRESENTATIONS_LOCAL_CSV

elif VIDEO_REPRESENTATION_SOURCE == AUTOENCODER_VIDEO_REPRESENTATION_SOURCE:
    VIDEO_REPRESENTATIONS_CSV = AUTOENCODER_LOCAL_VIDEO_REPRESENTATIONS_CSV

else:
    raise ValueError(
        f"Unsupported VIDEO_REPRESENTATION_SOURCE: "
        f"{VIDEO_REPRESENTATION_SOURCE}"
    )

# ------------------------------------------------------------
# Output artifact configuration
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Strict configuration validation
# ------------------------------------------------------------

required_config_values = [
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "TEXT_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATIONS_CSV",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "evaluation_split",
    "development_subset_size",
    "random_seed",
    "answer_mode",
    "choice_columns",
    "question_column",
    "video_id_column",
    "ground_truth_answer_column",
]

missing_config_values = [
    name for name in required_config_values
    if name not in globals()
]

if missing_config_values:
    raise NameError(
        "Missing required representation-based VideoQA configuration values: "
        + ", ".join(missing_config_values)
    )

if answer_mode != "multiple_choice":
    raise ValueError(
        f"Unsupported answer mode: {answer_mode}. "
        "Notebook 07 currently supports multiple_choice only."
    )

if not Path(TEXT_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing text representation artifact: {TEXT_REPRESENTATIONS_CSV}"
    )

if not Path(VIDEO_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing video representation artifact: {VIDEO_REPRESENTATIONS_CSV}"
    )

# ------------------------------------------------------------
# Display active configuration
# ------------------------------------------------------------

representation_videoqa_config_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_type": EXPERIMENT_TYPE,
    "dataset_mode": (
        "full_split"
        if RUN_FULL_EVALUATION_SPLIT
        else "development"
    ),
    "evaluation_split": evaluation_split,
    "development_subset_size": (
        "not applicable"
        if RUN_FULL_EVALUATION_SPLIT
        else development_subset_size
    ),
    "random_seed": random_seed,
    "answer_mode": answer_mode,
    "video_representation_source": VIDEO_REPRESENTATION_SOURCE,
    "text_representation_source": TEXT_REPRESENTATION_SOURCE,
    "prediction_method": REPRESENTATION_VIDEOQA_METHOD,
    "text_representation_file": str(TEXT_REPRESENTATIONS_CSV),
    "video_representation_file": str(VIDEO_REPRESENTATIONS_CSV),
    "local_output_directory": str(REPRESENTATION_VIDEOQA_LOCAL_DIR),
    "predictions_output_file": str(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    "validation_output_file": str(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    "summary_output_file": str(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
}

representation_videoqa_config_df = pd.DataFrame(
    representation_videoqa_config_summary.items(),
    columns=["Configuration Item", "Value"],
)

print("Representation-based VideoQA configuration defined.")
display(representation_videoqa_config_df)



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Verify that required runtime objects were initialized by previous steps.
* Confirm that the shared CLIP text and video representation artifacts are available.
* Validate the required NExT-QA annotation schema for representation-based VideoQA.
* Create and verify the local prediction output directory.
* Display runtime environment information and dependency versions.
* Confirm that the notebook is ready to load representation artifacts and perform VideoQA inference.

In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import platform
import numpy as np
import pandas as pd
from pathlib import Path

print("Verifying runtime environment and dependencies...")

# ------------------------------------------------------------
# Verify required in-memory objects
# ------------------------------------------------------------

required_runtime_objects = [
    "annotations_df",
    "split_summary_df",
    "TEXT_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATIONS_CSV",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "EXPERIMENT_NAME",
    "EXPERIMENT_TYPE",
    "TEXT_REPRESENTATION_SOURCE",
    "VIDEO_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_METHOD",
    "DEFAULT_TEXT_REPRESENTATION_SOURCE",
    "evaluation_split",
    "development_subset_size",
    "answer_mode",
    "choice_columns",
    "question_column",
    "video_id_column",
    "ground_truth_answer_column",
]

missing_runtime_objects = [
    name for name in required_runtime_objects
    if name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Missing required runtime objects: "
        + ", ".join(missing_runtime_objects)
    )

# ------------------------------------------------------------
# Verify Fusion MLP configuration
# ------------------------------------------------------------

if REPRESENTATION_VIDEOQA_METHOD != "fusion_mlp_classifier":
    raise ValueError(
        "Notebook 07 now requires REPRESENTATION_VIDEOQA_METHOD = "
        "'fusion_mlp_classifier'. "
        f"Found: {REPRESENTATION_VIDEOQA_METHOD}"
    )

if TEXT_REPRESENTATION_SOURCE != DEFAULT_TEXT_REPRESENTATION_SOURCE:
    raise ValueError(
        "Text representation source does not match configuration. "
        f"Found: {TEXT_REPRESENTATION_SOURCE}; "
        f"expected: {DEFAULT_TEXT_REPRESENTATION_SOURCE}"
    )

if VIDEO_REPRESENTATION_SOURCE not in {
    CLIP_VIDEO_REPRESENTATION_SOURCE,
    AUTOENCODER_VIDEO_REPRESENTATION_SOURCE,
    }:
    raise ValueError(
        f"Unsupported video representation source: "
        f"{VIDEO_REPRESENTATION_SOURCE}"
    )

if answer_mode != "multiple_choice":
    raise ValueError(
        f"Unsupported answer mode: {answer_mode}. "
        "Notebook 07 supports multiple_choice only."
    )

# ------------------------------------------------------------
# Verify artifact availability
# ------------------------------------------------------------

required_input_files = [
    Path(TEXT_REPRESENTATIONS_CSV),
    Path(VIDEO_REPRESENTATIONS_CSV),
]

missing_input_files = [
    path for path in required_input_files
    if not path.exists()
]

if missing_input_files:
    for path in missing_input_files:
        print(f"Missing input artifact: {path}")

    raise FileNotFoundError(
        "One or more required representation artifacts are missing."
    )

# ------------------------------------------------------------
# Verify required annotation columns
# ------------------------------------------------------------

required_annotation_columns = [
    "split",
    video_id_column,
    question_column,
    ground_truth_answer_column,
    *choice_columns,
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "annotations_df is missing required columns: "
        + ", ".join(missing_annotation_columns)
    )

# ------------------------------------------------------------
# Verify output directory
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not REPRESENTATION_VIDEOQA_LOCAL_DIR.exists():
    raise FileNotFoundError(
        f"Output directory was not created: {REPRESENTATION_VIDEOQA_LOCAL_DIR}"
    )

# ------------------------------------------------------------
# Display runtime summary
# ------------------------------------------------------------

runtime_summary = {
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "annotation_records": len(annotations_df),
    "text_representation_file": str(TEXT_REPRESENTATIONS_CSV),
    "video_representation_file": str(VIDEO_REPRESENTATIONS_CSV),
    "text_representation_file_exists": Path(TEXT_REPRESENTATIONS_CSV).exists(),
    "video_representation_file_exists": Path(VIDEO_REPRESENTATIONS_CSV).exists(),
    "output_directory": str(REPRESENTATION_VIDEOQA_LOCAL_DIR),
    "experiment_name": EXPERIMENT_NAME,
    "experiment_type": EXPERIMENT_TYPE,
    "prediction_method": REPRESENTATION_VIDEOQA_METHOD,
    "text_representation_source": TEXT_REPRESENTATION_SOURCE,
    "video_representation_source": VIDEO_REPRESENTATION_SOURCE,
}

runtime_summary_df = pd.DataFrame(
    runtime_summary.items(),
    columns=["Runtime Item", "Value"],
)

print("Runtime environment verification complete.")
display(runtime_summary_df)



### 🔷 Step 4 — Prepare NExT-QA Evaluation Dataset

* Select the configured NExT-QA evaluation split.
* Apply development-subset sampling when full-split execution is disabled.
* Reconstruct stable annotation identifiers matching the shared CLIP text representation dataset.
* Normalize video IDs, questions, answer choices, and ground-truth labels.
* Create stable representation-based QA record identifiers.
* Validate required fields, duplicate identifiers, and multiple-choice answer labels.
* Display the prepared QA input dataset for verification.

In [ ]:
# ============================================================
# Step 4: Prepare NExT-QA Evaluation Dataset
# ============================================================

import pandas as pd

print("Preparing NExT-QA evaluation dataset...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "annotations_df" not in globals():
    raise NameError("annotations_df was not found. Run Step 1 first.")

required_annotation_columns = [
    "split",
    "question_id",
    video_id_column,
    question_column,
    ground_truth_answer_column,
    *choice_columns,
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "annotations_df is missing required columns: "
        + ", ".join(missing_annotation_columns)
    )

# ------------------------------------------------------------
# Reconstruct Notebook 05 annotation_id values
# ------------------------------------------------------------

annotations_with_ids_df = annotations_df.copy()

annotations_with_ids_df["annotation_source_index"] = (
    annotations_with_ids_df.index
)

annotations_with_ids_df["annotation_id"] = (
    annotations_with_ids_df["split"].astype(str)
    + "_"
    + annotations_with_ids_df[video_id_column].astype(str)
    + "_"
    + annotations_with_ids_df["question_id"].astype(str)
    + "_"
    + annotations_with_ids_df["annotation_source_index"].astype(str)
)

# ------------------------------------------------------------
# Select evaluation split
# ------------------------------------------------------------

evaluation_annotations_df = annotations_with_ids_df[
    annotations_with_ids_df["split"] == evaluation_split
].copy()

if evaluation_annotations_df.empty:
    raise ValueError(
        f"No annotation records found for evaluation split: {evaluation_split}"
    )

# ------------------------------------------------------------
# Apply development subset if requested
# ------------------------------------------------------------

if RUN_FULL_EVALUATION_SPLIT:

    qa_input_df = (
        evaluation_annotations_df
        .sort_values([video_id_column, question_column])
        .reset_index(drop=True)
    )

else:

    development_video_ids = (
        evaluation_annotations_df[video_id_column]
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .head(development_subset_size)
        .tolist()
    )

    if not development_video_ids:
        raise RuntimeError(
            "No development videos were selected from evaluation annotations."
        )

    qa_input_df = (
        evaluation_annotations_df[
            evaluation_annotations_df[video_id_column]
            .astype(str)
            .isin(development_video_ids)
        ]
        .copy()
        .sort_values([video_id_column, question_column])
        .reset_index(drop=True)
    )

if qa_input_df.empty:
    raise RuntimeError(
        "No representation-based VideoQA input records were generated."
    )

# ------------------------------------------------------------
# Normalize key fields
# ------------------------------------------------------------

qa_input_df[video_id_column] = (
    qa_input_df[video_id_column]
    .astype(str)
)

qa_input_df[question_column] = (
    qa_input_df[question_column]
    .astype(str)
)

qa_input_df["annotation_id"] = (
    qa_input_df["annotation_id"]
    .astype(str)
)

for choice_column in choice_columns:
    qa_input_df[choice_column] = (
        qa_input_df[choice_column]
        .astype(str)
    )

qa_input_df[ground_truth_answer_column] = (
    qa_input_df[ground_truth_answer_column]
    .astype(int)
)

# ------------------------------------------------------------
# Add stable QA record id
# ------------------------------------------------------------

qa_input_df = qa_input_df.reset_index(drop=True)

qa_input_df["qa_record_id"] = (
    qa_input_df["annotation_id"].astype(str)
    + "_representation_videoqa"
)

if "NUM_MULTIPLE_CHOICE_ANSWERS" not in globals():
    raise NameError("NUM_MULTIPLE_CHOICE_ANSWERS was not found in configuration.")

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

duplicate_qa_ids = qa_input_df["qa_record_id"].duplicated().sum()

if duplicate_qa_ids > 0:
    raise ValueError(
        f"Found {duplicate_qa_ids} duplicate qa_record_id values."
    )

duplicate_annotation_ids = (
    qa_input_df["annotation_id"]
    .duplicated()
    .sum()
)

if duplicate_annotation_ids > 0:
    raise ValueError(
        f"Found {duplicate_annotation_ids} duplicate annotation_id values."
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer choices, "
        f"but found {len(choice_columns)}."
    )

invalid_ground_truth_values = sorted(
    set(qa_input_df[ground_truth_answer_column].unique())
    - set(range(NUM_MULTIPLE_CHOICE_ANSWERS))
)

if invalid_ground_truth_values:
    raise ValueError(
        "Found invalid ground-truth answer values: "
        + str(invalid_ground_truth_values)
    )

missing_required_values = (
    qa_input_df[
        [
            "qa_record_id",
            "annotation_id",
            "split",
            video_id_column,
            question_column,
            ground_truth_answer_column,
            *choice_columns,
        ]
    ]
    .isna()
    .sum()
    .sum()
)

if missing_required_values > 0:
    raise ValueError(
        f"Found {missing_required_values} missing required QA values."
    )

dataset_mode_label = (
    "full_split"
    if RUN_FULL_EVALUATION_SPLIT
    else "development"
)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("NExT-QA evaluation dataset prepared successfully.")
print(f"Dataset mode          : {dataset_mode_label}")
print(f"Evaluation split      : {evaluation_split}")
print(f"Source split records  : {len(evaluation_annotations_df):,}")
print(f"QA input records      : {len(qa_input_df):,}")
print(f"Unique videos         : {qa_input_df[video_id_column].nunique():,}")
print(f"Answer mode           : {answer_mode}")
print(f"Answer choices        : {len(choice_columns)}")

print("\nQA Input Preview:")
display(
    qa_input_df[
        [
            "qa_record_id",
            "annotation_id",
            "split",
            video_id_column,
            question_column,
            ground_truth_answer_column,
            *choice_columns,
        ]
    ].head(10)
)



### 🔷 Step 5 — Load CLIP Text Representations

* Load the shared CLIP text representation artifact generated by Notebook 05.
* Validate the required metadata and embedding columns.
* Identify the CLIP text embedding dimensions from the representation dataset.
* Verify that all embedding values are numeric, complete, and internally consistent.
* Confirm that all representation records have unique identifiers and valid input types.
* Filter the shared text representations to the videos and dataset split required for the current evaluation.
* Display a preview of the filtered CLIP text representation dataset for verification.

In [ ]:
# ============================================================
# Step 5: Load CLIP Text Representations
# ============================================================

import pandas as pd
from pathlib import Path

print("Loading CLIP text representations...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step5_objects = [
    "qa_input_df",
    "TEXT_REPRESENTATIONS_CSV",
    "TEXT_REPRESENTATION_SOURCE",
    "TEXT_INPUT_TYPES",
    "CLIP_TEXT_EMBEDDING_DIM",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "choice_columns",
]

missing_step5_objects = [
    name for name in required_step5_objects
    if name not in globals()
]

if missing_step5_objects:
    raise NameError(
        "Missing required Step 5 objects: "
        + ", ".join(missing_step5_objects)
    )

if not Path(TEXT_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing CLIP text representation file: {TEXT_REPRESENTATIONS_CSV}"
    )

# ------------------------------------------------------------
# Load text representation artifact
# ------------------------------------------------------------

clip_text_representation_df = pd.read_csv(TEXT_REPRESENTATIONS_CSV)

if clip_text_representation_df.empty:
    raise RuntimeError(
        "CLIP text representation artifact was loaded but is empty."
    )

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_text_columns = [
    "record_id",
    video_id_column,
    "question_id",
    "annotation_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "split",
    "representation_source",
]

missing_text_columns = [
    col for col in required_text_columns
    if col not in clip_text_representation_df.columns
]

if missing_text_columns:
    raise ValueError(
        "CLIP text representation dataset is missing required columns: "
        + ", ".join(missing_text_columns)
    )

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

clip_text_columns = sorted(
    [
        col
        for col in clip_text_representation_df.columns
        if col.startswith("clip_text_")
        and col.replace("clip_text_", "").isdigit()
    ],
    key=lambda col: int(col.replace("clip_text_", ""))
)

text_embedding_dimension = len(clip_text_columns)

if text_embedding_dimension == 0:
    raise ValueError("No CLIP text embedding columns were found.")

if text_embedding_dimension != CLIP_TEXT_EMBEDDING_DIM:
    raise ValueError(
        f"Expected {CLIP_TEXT_EMBEDDING_DIM} CLIP text dimensions, "
        f"found {text_embedding_dimension}."
    )

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

missing_embedding_values = (
    clip_text_representation_df[clip_text_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing CLIP text embedding values."
    )

non_numeric_embedding_columns = [
    col for col in clip_text_columns
    if not pd.api.types.is_numeric_dtype(clip_text_representation_df[col])
]

if non_numeric_embedding_columns:
    raise TypeError(
        "Found non-numeric CLIP text embedding columns: "
        + ", ".join(non_numeric_embedding_columns[:20])
    )

duplicate_text_record_ids = (
    clip_text_representation_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_text_record_ids > 0:
    raise ValueError(
        f"Found {duplicate_text_record_ids} duplicate CLIP text record_id values."
    )

expected_text_types = set(TEXT_INPUT_TYPES)

actual_text_types = set(
    clip_text_representation_df["text_type"]
    .astype(str)
    .unique()
)

unexpected_text_types = actual_text_types - expected_text_types

if unexpected_text_types:
    raise ValueError(
        "Unexpected CLIP text types found: "
        + ", ".join(sorted(unexpected_text_types))
    )

unexpected_sources = set(
    clip_text_representation_df["representation_source"]
    .astype(str)
    .unique()
) - {TEXT_REPRESENTATION_SOURCE}

if unexpected_sources:
    raise ValueError(
        "Unexpected text representation sources found: "
        + ", ".join(sorted(unexpected_sources))
    )

# ------------------------------------------------------------
# Filter to evaluation QA records
# ------------------------------------------------------------

qa_splits = set(qa_input_df["split"].astype(str).unique())

qa_annotation_ids = set(
    qa_input_df["annotation_id"]
    .astype(str)
    .unique()
)

clip_text_representation_df[video_id_column] = (
    clip_text_representation_df[video_id_column]
    .astype(str)
)

filtered_clip_text_df = clip_text_representation_df[
    clip_text_representation_df["annotation_id"]
    .astype(str)
    .isin(qa_annotation_ids)
    &
    clip_text_representation_df["split"]
    .astype(str)
    .isin(qa_splits)
].copy()

if filtered_clip_text_df.empty:
    raise RuntimeError(
        "No CLIP text representations matched the selected QA input records."
    )

expected_text_records = (
    len(qa_input_df)
    * (1 + NUM_MULTIPLE_CHOICE_ANSWERS)
)

if len(filtered_clip_text_df) != expected_text_records:
    raise ValueError(
        f"Expected {expected_text_records:,} filtered CLIP text "
        f"representation records, found {len(filtered_clip_text_df):,}."
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("CLIP text representations loaded successfully.")
print(f"Source file            : {TEXT_REPRESENTATIONS_CSV}")
print(f"Total text records     : {len(clip_text_representation_df):,}")
print(f"Filtered text records  : {len(filtered_clip_text_df):,}")
print(f"Text types             : {', '.join(sorted(actual_text_types))}")
print(f"Representation source  : {TEXT_REPRESENTATION_SOURCE}")
print(f"Embedding dimensions   : {text_embedding_dimension:,}")

print("\nFiltered CLIP Text Representation Preview:")
display(
    filtered_clip_text_df[
        [
            "record_id",
            video_id_column,
            "question_id",
            "annotation_id",
            "split",
            "text_type",
            "choice_index",
            "text",
            "representation_source",
        ]
    ].head(10)
)



### 🔷 Step 6 — Load Video Representations

* Load the configured video representation artifact for representation-based VideoQA.
* Validate required video representation metadata and embedding columns.
* Identify the video embedding dimension from the representation dataset.
* Verify that all video embeddings are numeric, complete, and uniquely keyed by video.
* Filter video representations to the videos required by the current QA input dataset.
* Confirm that every QA video has a matching video representation.
* Display a preview of the filtered video representation dataset for verification.

In [ ]:
# ============================================================
# Step 6: Load Video Representations
# ============================================================

import pandas as pd
from pathlib import Path
import re

print("Loading video representations...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step6_objects = [
    "qa_input_df",
    "VIDEO_REPRESENTATIONS_CSV",
    "VIDEO_REPRESENTATION_SOURCE",
    "video_id_column",
    "CLIP_VIDEO_EMBEDDING_DIM",
    "AUTOENCODER_VIDEO_EMBEDDING_DIM",
]

missing_step6_objects = [
    name for name in required_step6_objects
    if name not in globals()
]

if missing_step6_objects:
    raise NameError(
        "Missing required Step 6 objects: "
        + ", ".join(missing_step6_objects)
    )

if not Path(VIDEO_REPRESENTATIONS_CSV).exists():
    raise FileNotFoundError(
        f"Missing video representation file: {VIDEO_REPRESENTATIONS_CSV}"
    )

# ------------------------------------------------------------
# Load video representation artifact selected in Step 2
# ------------------------------------------------------------

video_representation_df = pd.read_csv(VIDEO_REPRESENTATIONS_CSV)

if video_representation_df.empty:
    raise RuntimeError(
        "Video representation artifact was loaded but is empty."
    )

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_video_representation_columns = [
    "record_id",
    video_id_column,
    "split",
    "representation_source",
]

missing_video_representation_columns = [
    col for col in required_video_representation_columns
    if col not in video_representation_df.columns
]

if missing_video_representation_columns:
    raise ValueError(
        "Video representation dataset is missing required columns: "
        + ", ".join(missing_video_representation_columns)
    )

# ------------------------------------------------------------
# Identify standardized embedding columns
# ------------------------------------------------------------

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

video_embedding_columns = sorted(
    [
        col for col in video_representation_df.columns
        if re.fullmatch(r"embedding_\d{3}", col)
    ],
    key=lambda col: int(col.replace("embedding_", ""))
)

if not video_embedding_columns and VIDEO_REPRESENTATION_SOURCE == CLIP_VIDEO_REPRESENTATION_SOURCE:

    legacy_clip_columns = sorted(
        [
            col for col in video_representation_df.columns
            if re.fullmatch(r"clip_video_\d{3}", col)
        ],
        key=lambda col: int(col.replace("clip_video_", ""))
    )

    if legacy_clip_columns:
        rename_map = {
            old_col: f"embedding_{idx:03d}"
            for idx, old_col in enumerate(legacy_clip_columns)
        }

        video_representation_df = video_representation_df.rename(
            columns=rename_map
        )

        video_embedding_columns = list(rename_map.values())

        print(
            "CLIP video embedding columns standardized from "
            "clip_video_### to embedding_###."
        )

if not video_embedding_columns:
    raise ValueError(
        "No video embedding columns found. Expected standardized "
        "embedding_### columns. For CLIP video artifacts, clip_video_### "
        "columns are also accepted and standardized."
    )

video_embedding_dimension = len(video_embedding_columns)

expected_video_embedding_dimension = (
    CLIP_VIDEO_EMBEDDING_DIM
    if VIDEO_REPRESENTATION_SOURCE == "clip_video"
    else AUTOENCODER_VIDEO_EMBEDDING_DIM
)

if video_embedding_dimension != expected_video_embedding_dimension:
    raise ValueError(
        f"Expected {expected_video_embedding_dimension} video embedding "
        f"dimensions for {VIDEO_REPRESENTATION_SOURCE}, "
        f"found {video_embedding_dimension}."
    )

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

video_representation_df[video_id_column] = (
    video_representation_df[video_id_column]
    .astype(str)
)

missing_embedding_values = (
    video_representation_df[video_embedding_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing video embedding values."
    )

non_numeric_embedding_columns = [
    col for col in video_embedding_columns
    if not pd.api.types.is_numeric_dtype(video_representation_df[col])
]

if non_numeric_embedding_columns:
    raise TypeError(
        "Found non-numeric video embedding columns: "
        + ", ".join(non_numeric_embedding_columns[:20])
    )

duplicate_video_records = (
    video_representation_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_video_records > 0:
    raise ValueError(
        f"Found {duplicate_video_records} duplicate video representation "
        "record_id values."
    )

duplicate_video_ids = (
    video_representation_df[video_id_column]
    .duplicated()
    .sum()
)

if duplicate_video_ids > 0:
    raise ValueError(
        f"Found {duplicate_video_ids} duplicate video representation records."
    )

actual_video_sources = set(
    video_representation_df["representation_source"]
    .astype(str)
    .unique()
)

if actual_video_sources != {VIDEO_REPRESENTATION_SOURCE}:
    raise ValueError(
        f"Expected only video representation source "
        f"{VIDEO_REPRESENTATION_SOURCE}, found: "
        + ", ".join(sorted(actual_video_sources))
    )

# ------------------------------------------------------------
# Filter to evaluation QA videos
# ------------------------------------------------------------

qa_video_ids = set(
    qa_input_df[video_id_column]
    .astype(str)
    .unique()
)

filtered_video_representation_df = video_representation_df[
    video_representation_df[video_id_column].isin(qa_video_ids)
].copy()

if filtered_video_representation_df.empty:
    raise RuntimeError(
        "No video representations matched the selected QA input records."
    )

missing_qa_video_ids = sorted(
    qa_video_ids
    - set(
        filtered_video_representation_df[video_id_column]
        .astype(str)
        .unique()
    )
)

if missing_qa_video_ids:
    raise ValueError(
        "Missing video representations for QA videos: "
        + ", ".join(missing_qa_video_ids[:20])
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("Video representations loaded successfully.")
print(f"Source file            : {VIDEO_REPRESENTATIONS_CSV}")
print(f"Total video records    : {len(video_representation_df):,}")
print(f"Filtered video records : {len(filtered_video_representation_df):,}")
print(f"QA input videos        : {len(qa_video_ids):,}")
print(f"Representation source  : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Embedding dimensions   : {video_embedding_dimension:,}")

print("\nFiltered Video Representation Preview:")
display(
    filtered_video_representation_df[
        [
            "record_id",
            video_id_column,
            "split",
            "representation_source",
        ]
    ].head(10)
)



### 🔷 Step 7 — Prepare Fusion MLP Training Dataset

* Load the question, answer-choice, and video representations required for inference.
* Associate each evaluation question with its corresponding video representation.
* Construct one candidate-answer record for each answer choice.
* Prepare multimodal feature vectors for Fusion MLP training and inference.
* Validate dataset completeness before classifier training.


In [ ]:
# ============================================================
# Step 7: Prepare Fusion MLP Training Dataset
# ============================================================

import pandas as pd

print("Preparing Fusion MLP training dataset...")

required_objects = [
    "qa_input_df",
    "filtered_clip_text_df",
    "filtered_video_representation_df",
    "clip_text_columns",
    "video_embedding_columns",
    "video_id_column",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "VIDEO_REPRESENTATION_SOURCE",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for representation QA preparation: "
        + ", ".join(missing_objects)
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} choices, "
        f"found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Normalize key fields
# ------------------------------------------------------------

qa_input_df["annotation_id"] = qa_input_df["annotation_id"].astype(str)
qa_input_df[video_id_column] = qa_input_df[video_id_column].astype(str)

filtered_clip_text_df["annotation_id"] = (
    filtered_clip_text_df["annotation_id"].astype(str)
)

filtered_video_representation_df[video_id_column] = (
    filtered_video_representation_df[video_id_column].astype(str)
)

# ------------------------------------------------------------
# Split text representations
# ------------------------------------------------------------

question_text_df = filtered_clip_text_df[
    filtered_clip_text_df["text_type"] == "question"
].copy()

answer_choice_text_df = filtered_clip_text_df[
    filtered_clip_text_df["text_type"] == "answer_choice"
].copy()

if question_text_df.empty:
    raise RuntimeError("No question text representations were found.")

if answer_choice_text_df.empty:
    raise RuntimeError("No answer-choice text representations were found.")

# ------------------------------------------------------------
# Validate expected text representation counts
# ------------------------------------------------------------

question_count_by_annotation = question_text_df.groupby("annotation_id").size()

invalid_question_counts = question_count_by_annotation[
    question_count_by_annotation != 1
]

if len(invalid_question_counts) > 0:
    raise ValueError(
        "Each QA record must have exactly one question representation. "
        f"Invalid records found: {len(invalid_question_counts)}"
    )

choice_count_by_annotation = answer_choice_text_df.groupby("annotation_id").size()

invalid_choice_counts = choice_count_by_annotation[
    choice_count_by_annotation != NUM_MULTIPLE_CHOICE_ANSWERS
]

if len(invalid_choice_counts) > 0:
    raise ValueError(
        "Each QA record must have exactly "
        f"{NUM_MULTIPLE_CHOICE_ANSWERS} answer-choice representations. "
        f"Invalid records found: {len(invalid_choice_counts)}"
    )

# ------------------------------------------------------------
# Build question representation lookup
# ------------------------------------------------------------

question_lookup_df = (
    question_text_df[
        [
            "annotation_id",
            "record_id",
            "text",
        ]
    ]
    .rename(
        columns={
            "record_id": "question_representation_id",
            "text": "question_text_from_representation",
        }
    )
)

# ------------------------------------------------------------
# Build video representation lookup
# ------------------------------------------------------------

video_lookup_df = (
    filtered_video_representation_df[
        [
            video_id_column,
            "record_id",
            "representation_source",
        ]
    ]
    .rename(
        columns={
            "record_id": "video_representation_id",
            "representation_source": "video_representation_source",
        }
    )
)

unexpected_video_sources = set(
    video_lookup_df["video_representation_source"].astype(str).unique()
) - {VIDEO_REPRESENTATION_SOURCE}

if unexpected_video_sources:
    raise ValueError(
        "Unexpected video representation sources found: "
        + ", ".join(sorted(unexpected_video_sources))
    )

# ------------------------------------------------------------
# Build one row per answer choice
# ------------------------------------------------------------

answer_choice_input_df = (
    answer_choice_text_df[
        [
            "annotation_id",
            "record_id",
            "choice_index",
            "text",
        ]
    ]
    .rename(
        columns={
            "record_id": "answer_choice_representation_id",
            "text": "answer_choice_text_from_representation",
        }
    )
    .copy()
)

answer_choice_input_df["choice_index"] = (
    answer_choice_input_df["choice_index"].astype(int)
)

representation_qa_df = (
    qa_input_df
    .merge(question_lookup_df, on="annotation_id", how="left")
    .merge(video_lookup_df, on=video_id_column, how="left")
    .merge(answer_choice_input_df, on="annotation_id", how="left")
)

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

expected_candidate_rows = (
    len(qa_input_df)
    * NUM_MULTIPLE_CHOICE_ANSWERS
)

if len(representation_qa_df) != expected_candidate_rows:
    raise ValueError(
        f"Expected {expected_candidate_rows:,} representation QA rows, "
        f"found {len(representation_qa_df):,}."
    )

required_representation_qa_columns = [
    "qa_record_id",
    "annotation_id",
    "split",
    video_id_column,
    question_column,
    ground_truth_answer_column,
    "question_representation_id",
    "video_representation_id",
    "video_representation_source",
    "answer_choice_representation_id",
    "choice_index",
    "answer_choice_text_from_representation",
]

missing_representation_qa_columns = [
    col for col in required_representation_qa_columns
    if col not in representation_qa_df.columns
]

if missing_representation_qa_columns:
    raise ValueError(
        "representation_qa_df is missing required columns: "
        + ", ".join(missing_representation_qa_columns)
    )

missing_required_values = (
    representation_qa_df[required_representation_qa_columns]
    .isna()
    .sum()
    .sum()
)

if missing_required_values > 0:
    raise ValueError(
        f"Found {missing_required_values} missing required values "
        "in representation_qa_df."
    )

candidate_count_by_qa = representation_qa_df.groupby("qa_record_id").size()

invalid_candidate_counts = candidate_count_by_qa[
    candidate_count_by_qa != NUM_MULTIPLE_CHOICE_ANSWERS
]

if len(invalid_candidate_counts) > 0:
    raise ValueError(
        "Each QA record must have exactly "
        f"{NUM_MULTIPLE_CHOICE_ANSWERS} candidate answers. "
        f"Invalid QA records found: {len(invalid_candidate_counts)}"
    )

duplicate_candidate_rows = (
    representation_qa_df[
        [
            "qa_record_id",
            "choice_index",
        ]
    ]
    .duplicated()
    .sum()
)

if duplicate_candidate_rows > 0:
    raise ValueError(
        f"Found {duplicate_candidate_rows} duplicate QA/choice rows."
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("Representation-based QA dataset prepared successfully.")
print(f"QA records              : {len(qa_input_df):,}")
print(f"Candidate answer rows   : {len(representation_qa_df):,}")
print(f"Choices per question    : {NUM_MULTIPLE_CHOICE_ANSWERS}")
print(f"Unique videos           : {representation_qa_df[video_id_column].nunique():,}")
print(f"Video source            : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text embedding dim      : {len(clip_text_columns):,}")
print(f"Video embedding dim     : {len(video_embedding_columns):,}")

print("\nRepresentation QA Preview:")
display(
    representation_qa_df[
        [
            "qa_record_id",
            "annotation_id",
            video_id_column,
            question_column,
            ground_truth_answer_column,
            "choice_index",
            "answer_choice_text_from_representation",
            "question_representation_id",
            "video_representation_id",
            "video_representation_source",
            "answer_choice_representation_id",
        ]
    ].head(10)
)



### 🔷 Step 8 — Train Fusion MLP Classifier and Generate Predictions

* Build multimodal feature vectors from video, question, and answer representations.
* Project each representation into the shared embedding space.
* Fuse the projected representations using the Fusion MLP classifier.
* Train the classifier using CrossEntropyLoss on the configured evaluation dataset.
* Score each candidate answer and select the highest-scoring multiple-choice prediction.
* Record predictions together with experiment metadata and runtime statistics.
* Generate the prediction artifacts used for downstream evaluation.


In [ ]:
# ============================================================
# Step 8: Train Fusion MLP Classifier and Generate Predictions
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Generating Fusion MLP multiple-choice predictions...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_objects = [
    "representation_qa_df",
    "filtered_clip_text_df",
    "filtered_video_representation_df",
    "clip_text_columns",
    "video_embedding_columns",
    "REPRESENTATION_VIDEOQA_METHOD",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "FUSION_EMBEDDING_DIM",
    "FUSION_HIDDEN_DIM_1",
    "FUSION_HIDDEN_DIM_2",
    "FUSION_OUTPUT_DIM",
    "FUSION_DROPOUT",
    "FUSION_BATCH_SIZE",
    "FUSION_LEARNING_RATE",
    "FUSION_WEIGHT_DECAY",
    "FUSION_EPOCHS",
    "FUSION_RANDOM_SEED",
    "USE_LAYER_NORMALIZATION",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for Fusion MLP prediction generation: "
        + ", ".join(missing_objects)
    )

if REPRESENTATION_VIDEOQA_METHOD != "fusion_mlp_classifier":
    raise ValueError(
        f"Unsupported prediction method: {REPRESENTATION_VIDEOQA_METHOD}"
    )

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

np.random.seed(FUSION_RANDOM_SEED)
torch.manual_seed(FUSION_RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text_embedding_dimension = len(clip_text_columns)
video_embedding_dimension = len(video_embedding_columns)

# ------------------------------------------------------------
# Build embedding lookup dictionaries
# ------------------------------------------------------------

text_embedding_lookup = {}

for row in filtered_clip_text_df.to_dict("records"):
    text_embedding_lookup[row["record_id"]] = np.array(
        [row[col] for col in clip_text_columns],
        dtype=np.float32,
    )

video_embedding_lookup = {}

for row in filtered_video_representation_df.to_dict("records"):
    video_embedding_lookup[row["record_id"]] = np.array(
        [row[col] for col in video_embedding_columns],
        dtype=np.float32,
    )

# ------------------------------------------------------------
# Build one tensor sample per QA record
# ------------------------------------------------------------

qa_samples = []

for qa_record_id, group_df in representation_qa_df.groupby("qa_record_id"):

    group_df = group_df.sort_values("choice_index").reset_index(drop=True)

    if len(group_df) != NUM_MULTIPLE_CHOICE_ANSWERS:
        raise ValueError(
            f"QA record {qa_record_id} has {len(group_df)} candidates, "
            f"expected {NUM_MULTIPLE_CHOICE_ANSWERS}."
        )

    first_row = group_df.iloc[0]

    ground_truth_choice = int(first_row[ground_truth_answer_column])

    if ground_truth_choice not in range(NUM_MULTIPLE_CHOICE_ANSWERS):
        raise ValueError(
            f"Invalid ground-truth choice for {qa_record_id}: "
            f"{ground_truth_choice}"
        )

    question_id = first_row["question_representation_id"]
    video_id = first_row["video_representation_id"]

    if question_id not in text_embedding_lookup:
        raise KeyError(f"Missing question embedding: {question_id}")

    if video_id not in video_embedding_lookup:
        raise KeyError(f"Missing video embedding: {video_id}")

    question_embedding = text_embedding_lookup[question_id]
    video_embedding = video_embedding_lookup[video_id]

    answer_embeddings = []

    for _, row in group_df.iterrows():
        answer_id = row["answer_choice_representation_id"]

        if answer_id not in text_embedding_lookup:
            raise KeyError(f"Missing answer-choice embedding: {answer_id}")

        answer_embeddings.append(text_embedding_lookup[answer_id])

    qa_samples.append(
        {
            "qa_record_id": qa_record_id,
            "metadata": first_row.to_dict(),
            "candidate_metadata": group_df.to_dict("records"),
            "video_embedding": video_embedding,
            "question_embedding": question_embedding,
            "answer_embeddings": np.stack(answer_embeddings),
            "label": ground_truth_choice,
        }
    )

if not qa_samples:
    raise RuntimeError("No QA samples were created for Fusion MLP.")

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class FusionQADataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        return {
            "video": torch.tensor(sample["video_embedding"], dtype=torch.float32),
            "question": torch.tensor(sample["question_embedding"], dtype=torch.float32),
            "answers": torch.tensor(sample["answer_embeddings"], dtype=torch.float32),
            "label": torch.tensor(sample["label"], dtype=torch.long),
        }


fusion_dataset = FusionQADataset(qa_samples)

fusion_loader = DataLoader(
    fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=True,
)

# ------------------------------------------------------------
# Fusion MLP model
# ------------------------------------------------------------

class FusionMLPClassifier(nn.Module):
    def __init__(
        self,
        video_input_dim,
        text_input_dim,
        fusion_embedding_dim,
        hidden_dim_1,
        hidden_dim_2,
        output_dim,
        dropout,
        use_layer_norm=True,
    ):
        super().__init__()

        self.video_projection = nn.Linear(
            video_input_dim,
            fusion_embedding_dim,
        )

        self.text_projection = nn.Linear(
            text_input_dim,
            fusion_embedding_dim,
        )

        fusion_input_dim = fusion_embedding_dim * 3

        layers = [
            nn.Linear(fusion_input_dim, hidden_dim_1),
            nn.ReLU(),
        ]

        if use_layer_norm:
            layers.append(nn.LayerNorm(hidden_dim_1))

        layers.extend(
            [
                nn.Dropout(dropout),
                nn.Linear(hidden_dim_1, hidden_dim_2),
                nn.ReLU(),
            ]
        )

        if use_layer_norm:
            layers.append(nn.LayerNorm(hidden_dim_2))

        layers.extend(
            [
                nn.Dropout(dropout),
                nn.Linear(hidden_dim_2, output_dim),
            ]
        )

        self.scorer = nn.Sequential(*layers)

    def forward(self, video, question, answers):
        batch_size, num_choices, _ = answers.shape

        video_projected = self.video_projection(video)
        question_projected = self.text_projection(question)
        answers_projected = self.text_projection(answers)

        video_expanded = video_projected.unsqueeze(1).expand(
            -1,
            num_choices,
            -1,
        )

        question_expanded = question_projected.unsqueeze(1).expand(
            -1,
            num_choices,
            -1,
        )

        fused = torch.cat(
            [
                video_expanded,
                question_expanded,
                answers_projected,
            ],
            dim=-1,
        )

        scores = self.scorer(fused).squeeze(-1)

        return scores


fusion_model = FusionMLPClassifier(
    video_input_dim=video_embedding_dimension,
    text_input_dim=text_embedding_dimension,
    fusion_embedding_dim=FUSION_EMBEDDING_DIM,
    hidden_dim_1=FUSION_HIDDEN_DIM_1,
    hidden_dim_2=FUSION_HIDDEN_DIM_2,
    output_dim=FUSION_OUTPUT_DIM,
    dropout=FUSION_DROPOUT,
    use_layer_norm=USE_LAYER_NORMALIZATION,
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    fusion_model.parameters(),
    lr=FUSION_LEARNING_RATE,
    weight_decay=FUSION_WEIGHT_DECAY,
)

# ------------------------------------------------------------
# Train Fusion MLP
# ------------------------------------------------------------

fusion_model.train()

training_history = []

for epoch in range(FUSION_EPOCHS):

    epoch_losses = []

    for batch in fusion_loader:

        video_batch = batch["video"].to(device)
        question_batch = batch["question"].to(device)
        answers_batch = batch["answers"].to(device)
        label_batch = batch["label"].to(device)

        optimizer.zero_grad()

        scores = fusion_model(
            video_batch,
            question_batch,
            answers_batch,
        )

        loss = criterion(scores, label_batch)

        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    mean_epoch_loss = float(np.mean(epoch_losses))

    training_history.append(
        {
            "epoch": epoch + 1,
            "loss": mean_epoch_loss,
        }
    )

fusion_training_history_df = pd.DataFrame(training_history)

# ------------------------------------------------------------
# Inference
# ------------------------------------------------------------

fusion_model.eval()

all_scores = []

inference_loader = DataLoader(
    fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=False,
)

with torch.no_grad():

    for batch in inference_loader:

        video_batch = batch["video"].to(device)
        question_batch = batch["question"].to(device)
        answers_batch = batch["answers"].to(device)

        scores = fusion_model(
            video_batch,
            question_batch,
            answers_batch,
        )

        all_scores.append(scores.cpu().numpy())

all_scores = np.vstack(all_scores)

# ------------------------------------------------------------
# Build scored candidate dataframe
# ------------------------------------------------------------

scored_candidate_records = []
prediction_rows = []

for sample_index, sample in enumerate(qa_samples):

    scores = all_scores[sample_index]
    predicted_choice = int(np.argmax(scores))
    ground_truth_choice = int(sample["label"])
    first_row = sample["metadata"]

    for candidate_row in sample["candidate_metadata"]:

        choice_index = int(candidate_row["choice_index"])

        scored_record = dict(candidate_row)
        scored_record["prediction_score"] = float(scores[choice_index])
        scored_record["prediction_method"] = REPRESENTATION_VIDEOQA_METHOD
        scored_record["text_embedding_dimension"] = text_embedding_dimension
        scored_record["video_embedding_dimension"] = video_embedding_dimension
        scored_record["fusion_embedding_dimension"] = FUSION_EMBEDDING_DIM

        scored_candidate_records.append(scored_record)

    prediction_rows.append(
        {
            "qa_record_id": sample["qa_record_id"],
            "annotation_id": first_row["annotation_id"],
            "split": first_row["split"],
            "video": first_row[video_id_column],
            "question": first_row[question_column],
            "ground_truth_choice": ground_truth_choice,
            "predicted_choice": predicted_choice,
            "choice_correct": predicted_choice == ground_truth_choice,
            "ground_truth": first_row[f"a{ground_truth_choice}"],
            "prediction": sample["candidate_metadata"][predicted_choice][
                "answer_choice_text_from_representation"
            ],
            "prediction_score": float(scores[predicted_choice]),
            "prediction_method": REPRESENTATION_VIDEOQA_METHOD,
            "video_representation_source": VIDEO_REPRESENTATION_SOURCE,
            "text_representation_source": TEXT_REPRESENTATION_SOURCE,
            "text_embedding_dimension": text_embedding_dimension,
            "video_embedding_dimension": video_embedding_dimension,
            "fusion_embedding_dimension": FUSION_EMBEDDING_DIM,
        }
    )

scored_candidate_df = pd.DataFrame(scored_candidate_records)
representation_prediction_df = pd.DataFrame(prediction_rows)

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

expected_prediction_records = len(qa_input_df)

if len(representation_prediction_df) != expected_prediction_records:
    raise ValueError(
        f"Expected {expected_prediction_records:,} prediction records, "
        f"found {len(representation_prediction_df):,}."
    )

invalid_predicted_choices = sorted(
    set(representation_prediction_df["predicted_choice"].unique())
    - set(range(NUM_MULTIPLE_CHOICE_ANSWERS))
)

if invalid_predicted_choices:
    raise ValueError(
        "Found invalid predicted choice values: "
        + str(invalid_predicted_choices)
    )

missing_prediction_values = (
    representation_prediction_df
    .isna()
    .sum()
    .sum()
)

if missing_prediction_values > 0:
    raise ValueError(
        f"Found {missing_prediction_values} missing prediction values."
    )

representation_choice_accuracy = (
    representation_prediction_df["choice_correct"].mean()
)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("Fusion MLP predictions generated successfully.")
print(f"Device                  : {device}")
print(f"QA records              : {len(representation_prediction_df):,}")
print(f"Candidate rows scored   : {len(scored_candidate_df):,}")
print(f"Prediction method       : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Text embedding dim      : {text_embedding_dimension}")
print(f"Video embedding dim     : {video_embedding_dimension}")
print(f"Fusion embedding dim    : {FUSION_EMBEDDING_DIM}")
print(f"Epochs                  : {FUSION_EPOCHS}")
print(f"Final training loss     : {fusion_training_history_df['loss'].iloc[-1]:.4f}")
print(f"Correct predictions     : {representation_prediction_df['choice_correct'].sum():,}")
print(f"Choice accuracy         : {representation_choice_accuracy:.3f}")

print("\nPrediction Preview:")
display(
    representation_prediction_df[
        [
            "qa_record_id",
            "video",
            "question",
            "ground_truth_choice",
            "predicted_choice",
            "choice_correct",
            "ground_truth",
            "prediction",
            "prediction_score",
        ]
    ].head(10)
)



### 🔷 Step 9 — Validate Prediction Dataset

* Verify that the generated prediction dataset contains the expected number of QA prediction records.
* Confirm that one prediction exists for every evaluation question and that all candidate-answer rows were processed.
* Validate required prediction fields, duplicate identifiers, and missing values.
* Verify that all ground-truth and predicted answer indices are valid multiple-choice labels.
* Confirm that prediction correctness values are valid Boolean indicators.
* Display a validation summary confirming the integrity of the representation-based prediction dataset.

In [ ]:
# ============================================================
# Step 9: Validate Prediction Dataset
# ============================================================

import pandas as pd
import numpy as np

print("Validating representation-based prediction dataset...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step9_objects = [
    "representation_prediction_df",
    "scored_candidate_df",
    "qa_input_df",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
]

missing_step9_objects = [
    name for name in required_step9_objects
    if name not in globals()
]

if missing_step9_objects:
    raise NameError(
        "Missing required Step 9 objects: "
        + ", ".join(missing_step9_objects)
    )

# ------------------------------------------------------------
# Validate required prediction columns
# ------------------------------------------------------------

required_prediction_columns = [
    "qa_record_id",
    "annotation_id",
    "split",
    "video",
    "question",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "ground_truth",
    "prediction",
    "prediction_score",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
]

missing_prediction_columns = [
    col for col in required_prediction_columns
    if col not in representation_prediction_df.columns
]

if missing_prediction_columns:
    raise ValueError(
        "representation_prediction_df is missing required columns: "
        + ", ".join(missing_prediction_columns)
    )

if len(choice_columns) != NUM_MULTIPLE_CHOICE_ANSWERS:
    raise ValueError(
        f"Expected {NUM_MULTIPLE_CHOICE_ANSWERS} answer choices, "
        f"found {len(choice_columns)}."
    )

# ------------------------------------------------------------
# Compute validation checks
# ------------------------------------------------------------

validation_checks = {
    "prediction_records": len(representation_prediction_df),
    "expected_prediction_records": len(qa_input_df),
    "candidate_rows": len(scored_candidate_df),
    "expected_candidate_rows": len(qa_input_df) * NUM_MULTIPLE_CHOICE_ANSWERS,
    "unique_qa_records": representation_prediction_df["qa_record_id"].nunique(),
    "unique_videos": representation_prediction_df["video"].nunique(),
    "missing_values": int(
        representation_prediction_df[required_prediction_columns]
        .isna()
        .sum()
        .sum()
    ),
    "duplicate_qa_record_ids": int(
        representation_prediction_df["qa_record_id"]
        .duplicated()
        .sum()
    ),
    "invalid_ground_truth_choices": int(
        (~representation_prediction_df["ground_truth_choice"]
         .isin(range(NUM_MULTIPLE_CHOICE_ANSWERS)))
        .sum()
    ),
    "invalid_predicted_choices": int(
        (~representation_prediction_df["predicted_choice"]
         .isin(range(NUM_MULTIPLE_CHOICE_ANSWERS)))
        .sum()
    ),
    "non_boolean_choice_correct": int(
        (~representation_prediction_df["choice_correct"]
         .map(lambda value: isinstance(value, (bool, np.bool_))))
        .sum()
    ),
    "invalid_prediction_method": int(
        (representation_prediction_df["prediction_method"]
         != REPRESENTATION_VIDEOQA_METHOD)
        .sum()
    ),
    "invalid_video_representation_source": int(
        (representation_prediction_df["video_representation_source"]
         != VIDEO_REPRESENTATION_SOURCE)
        .sum()
    ),
    "invalid_text_representation_source": int(
        (representation_prediction_df["text_representation_source"]
         != TEXT_REPRESENTATION_SOURCE)
        .sum()
    ),
}

validation_summary_df = pd.DataFrame(
    validation_checks.items(),
    columns=["Validation Check", "Value"],
)

# ------------------------------------------------------------
# Fail on validation errors
# ------------------------------------------------------------

if validation_checks["prediction_records"] != validation_checks["expected_prediction_records"]:
    raise ValueError(
        "Prediction record count does not match expected QA input count."
    )

if validation_checks["candidate_rows"] != validation_checks["expected_candidate_rows"]:
    raise ValueError(
        "Candidate row count does not match expected QA-choice count."
    )

if validation_checks["unique_qa_records"] != validation_checks["prediction_records"]:
    raise ValueError(
        "Prediction dataset does not contain exactly one row per QA record."
    )

if validation_checks["missing_values"] > 0:
    raise ValueError(
        f"Prediction dataset contains {validation_checks['missing_values']} missing values."
    )

if validation_checks["duplicate_qa_record_ids"] > 0:
    raise ValueError(
        f"Prediction dataset contains "
        f"{validation_checks['duplicate_qa_record_ids']} duplicate QA record IDs."
    )

if validation_checks["invalid_ground_truth_choices"] > 0:
    raise ValueError(
        "Prediction dataset contains invalid ground-truth choice labels."
    )

if validation_checks["invalid_predicted_choices"] > 0:
    raise ValueError(
        "Prediction dataset contains invalid predicted choice labels."
    )

if validation_checks["non_boolean_choice_correct"] > 0:
    raise ValueError(
        "Prediction dataset contains non-boolean choice_correct values."
    )

if validation_checks["invalid_prediction_method"] > 0:
    raise ValueError(
        "Prediction dataset contains unexpected prediction_method values."
    )

if validation_checks["invalid_video_representation_source"] > 0:
    raise ValueError(
        "Prediction dataset contains unexpected video_representation_source values."
    )

if validation_checks["invalid_text_representation_source"] > 0:
    raise ValueError(
        "Prediction dataset contains unexpected text_representation_source values."
    )

# ------------------------------------------------------------
# Display validation summary
# ------------------------------------------------------------

print("Prediction dataset validation passed.")
display(validation_summary_df)



### 🔷 Step 10 — Save Representation-Based VideoQA Results

* Create the configured output directory for the current representation-based VideoQA experiment.
* Save the prediction dataset and validation summary to CSV files.
* Verify that both output files were created successfully.
* Reload the saved files to confirm successful persistence and record integrity.
* Display the output locations and summary of the saved prediction and validation artifacts.

In [ ]:
# ============================================================
# Step 10: Save Representation-Based VideoQA Results
# ============================================================

import pandas as pd
import shutil
from pathlib import Path

print("Saving representation-based VideoQA results...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_objects = [
    "representation_prediction_df",
    "validation_summary_df",
    "representation_choice_accuracy",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "FUSION_EMBEDDING_DIM",
    "FUSION_EPOCHS",
    "fusion_training_history_df",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects for saving results: "
        + ", ".join(missing_objects)
    )

if fusion_training_history_df.empty:
    raise ValueError("fusion_training_history_df is empty.")

if "loss" not in fusion_training_history_df.columns:
    raise ValueError("fusion_training_history_df is missing required column: loss")

# ------------------------------------------------------------
# Build summary dataset
# ------------------------------------------------------------

representation_summary_df = pd.DataFrame(
    [
        {
            "pipeline": "representation_videoqa",
            "experiment_name": EXPERIMENT_NAME,
            "experiment_type": EXPERIMENT_TYPE,
            "prediction_method": REPRESENTATION_VIDEOQA_METHOD,
            "video_representation_source": VIDEO_REPRESENTATION_SOURCE,
            "text_representation_source": TEXT_REPRESENTATION_SOURCE,
            "evaluation_samples": len(representation_prediction_df),
            "unique_questions": representation_prediction_df["qa_record_id"].nunique(),
            "unique_videos": representation_prediction_df["video"].nunique(),
            "valid_choice_predictions": len(representation_prediction_df),
            "correct_choice_predictions": int(
                representation_prediction_df["choice_correct"].sum()
            ),
            "choice_accuracy": float(representation_choice_accuracy),
            "text_embedding_dimension": int(text_embedding_dimension),
            "video_embedding_dimension": int(video_embedding_dimension),
            "fusion_embedding_dimension": int(FUSION_EMBEDDING_DIM),
            "training_epochs": int(FUSION_EPOCHS),
            "final_training_loss": float(
                fusion_training_history_df["loss"].iloc[-1]
            ),
        }
    ]
)

# ------------------------------------------------------------
# Create local output directory
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

if not REPRESENTATION_VIDEOQA_LOCAL_DIR.exists():
    raise FileNotFoundError(
        f"Output directory was not created: {REPRESENTATION_VIDEOQA_LOCAL_DIR}"
    )

# ------------------------------------------------------------
# Save local artifacts
# ------------------------------------------------------------

representation_prediction_df.to_csv(
    REPRESENTATION_VIDEOQA_PREDICTIONS_CSV,
    index=False,
)

validation_summary_df.to_csv(
    REPRESENTATION_VIDEOQA_VALIDATION_CSV,
    index=False,
)

representation_summary_df.to_csv(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV,
    index=False,
)

local_artifact_checks = [
    REPRESENTATION_VIDEOQA_PREDICTIONS_CSV,
    REPRESENTATION_VIDEOQA_VALIDATION_CSV,
    REPRESENTATION_VIDEOQA_SUMMARY_CSV,
]

for local_path in local_artifact_checks:
    if not Path(local_path).exists():
        raise FileNotFoundError(
            f"Failed to create local artifact: {local_path}"
        )

# ------------------------------------------------------------
# Verify saved local files
# ------------------------------------------------------------

saved_predictions_check_df = pd.read_csv(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV)

if len(saved_predictions_check_df) != len(representation_prediction_df):
    raise ValueError(
        "Saved prediction file row count does not match "
        "the in-memory prediction dataset."
    )

saved_validation_check_df = pd.read_csv(REPRESENTATION_VIDEOQA_VALIDATION_CSV)

if len(saved_validation_check_df) != len(validation_summary_df):
    raise ValueError(
        "Saved validation file row count does not match "
        "the in-memory validation summary."
    )

saved_summary_check_df = pd.read_csv(REPRESENTATION_VIDEOQA_SUMMARY_CSV)

if len(saved_summary_check_df) != len(representation_summary_df):
    raise ValueError(
        "Saved summary file row count does not match "
        "the in-memory summary dataset."
    )

required_summary_columns = [
    "pipeline",
    "experiment_name",
    "experiment_type",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "evaluation_samples",
    "unique_questions",
    "unique_videos",
    "valid_choice_predictions",
    "correct_choice_predictions",
    "choice_accuracy",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
    "training_epochs",
    "final_training_loss",
]

missing_saved_summary_columns = [
    col for col in required_summary_columns
    if col not in saved_summary_check_df.columns
]

if missing_saved_summary_columns:
    raise ValueError(
        "Saved summary file is missing required columns: "
        + ", ".join(missing_saved_summary_columns)
    )

# ------------------------------------------------------------
# Promote artifacts to Google Drive
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

artifact_pairs = [
    (
        REPRESENTATION_VIDEOQA_PREDICTIONS_CSV,
        REPRESENTATION_VIDEOQA_PREDICTIONS_DRIVE_CSV,
    ),
    (
        REPRESENTATION_VIDEOQA_VALIDATION_CSV,
        REPRESENTATION_VIDEOQA_VALIDATION_DRIVE_CSV,
    ),
    (
        REPRESENTATION_VIDEOQA_SUMMARY_CSV,
        REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV,
    ),
]

for local_path, drive_path in artifact_pairs:
    shutil.copy2(local_path, drive_path)

    if not Path(drive_path).exists():
        raise FileNotFoundError(
            f"Failed to promote artifact: {drive_path}"
        )

print("Artifacts promoted to Google Drive.")

# ------------------------------------------------------------
# Display save summary
# ------------------------------------------------------------

print("Representation-based VideoQA results saved successfully.")
print(f"Local output directory : {REPRESENTATION_VIDEOQA_LOCAL_DIR}")
print(f"Drive output directory : {REPRESENTATION_VIDEOQA_DRIVE_DIR}")
print(f"Prediction file        : {REPRESENTATION_VIDEOQA_PREDICTIONS_CSV}")
print(f"Validation file        : {REPRESENTATION_VIDEOQA_VALIDATION_CSV}")
print(f"Summary file           : {REPRESENTATION_VIDEOQA_SUMMARY_CSV}")
print(f"Prediction records     : {len(representation_prediction_df):,}")
print(f"Validation checks      : {len(validation_summary_df):,}")
print(f"Summary records        : {len(representation_summary_df):,}")



### 🔷 Step 11 — Generate Prediction Summary Report

* Compute summary statistics describing the completed representation-based VideoQA experiment.
* Record the experiment configuration, representation sources, Fusion MLP classifier, and embedding dimensions.
* Summarize prediction counts, correct predictions, incorrect predictions, and multiple-choice accuracy.
* Save the experiment summary as a reusable artifact for downstream evaluation and comparison.
* Verify successful creation of the experiment summary report.


In [ ]:
# ============================================================
# Step 11: Generate Prediction Summary Report
# ============================================================

import pandas as pd
import shutil
from pathlib import Path

print("Generating representation-based VideoQA summary report...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step11_objects = [
    "representation_prediction_df",
    "scored_candidate_df",
    "validation_summary_df",
    "RUN_FULL_EVALUATION_SPLIT",
    "evaluation_split",
    "answer_mode",
    "choice_columns",
    "NUM_MULTIPLE_CHOICE_ANSWERS",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
    "REPRESENTATION_VIDEOQA_LOCAL_DIR",
    "REPRESENTATION_VIDEOQA_DRIVE_DIR",
    "REPRESENTATION_VIDEOQA_PREDICTIONS_CSV",
    "REPRESENTATION_VIDEOQA_VALIDATION_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_CSV",
    "REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV",
]

missing_step11_objects = [
    name for name in required_step11_objects
    if name not in globals()
]

if missing_step11_objects:
    raise NameError(
        "Missing required Step 11 objects: "
        + ", ".join(missing_step11_objects)
    )

# ------------------------------------------------------------
# Compute summary metrics
# ------------------------------------------------------------

correct_predictions = int(
    representation_prediction_df["choice_correct"].sum()
)

prediction_records = len(representation_prediction_df)

choice_accuracy = (
    correct_predictions / prediction_records
    if prediction_records > 0
    else 0.0
)

dataset_mode_label = (
    "full_split"
    if RUN_FULL_EVALUATION_SPLIT
    else "development"
)

summary_rows = [
    {"metric": "dataset_mode", "value": dataset_mode_label},
    {"metric": "evaluation_split", "value": evaluation_split},
    {"metric": "answer_mode", "value": answer_mode},
    {"metric": "prediction_method", "value": REPRESENTATION_VIDEOQA_METHOD},
    {"metric": "video_representation_source", "value": VIDEO_REPRESENTATION_SOURCE},
    {"metric": "text_representation_source", "value": TEXT_REPRESENTATION_SOURCE},
    {"metric": "qa_records", "value": prediction_records},
    {"metric": "candidate_rows", "value": len(scored_candidate_df)},
    {"metric": "choice_count", "value": NUM_MULTIPLE_CHOICE_ANSWERS},
    {"metric": "correct_predictions", "value": correct_predictions},
    {"metric": "choice_accuracy", "value": choice_accuracy},
    {
        "metric": "text_embedding_dimension",
        "value": int(representation_prediction_df["text_embedding_dimension"].iloc[0]),
    },
    {
        "metric": "video_embedding_dimension",
        "value": int(representation_prediction_df["video_embedding_dimension"].iloc[0]),
    },
    {
        "metric": "fusion_embedding_dimension",
        "value": int(representation_prediction_df["fusion_embedding_dimension"].iloc[0]),
    },
    {
        "metric": "unique_videos",
        "value": representation_prediction_df["video"].nunique(),
    },
    {
        "metric": "prediction_file",
        "value": str(REPRESENTATION_VIDEOQA_PREDICTIONS_CSV),
    },
    {
        "metric": "validation_file",
        "value": str(REPRESENTATION_VIDEOQA_VALIDATION_CSV),
    },
    {
        "metric": "summary_file",
        "value": str(REPRESENTATION_VIDEOQA_SUMMARY_CSV),
    },
]

representation_videoqa_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report locally
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

representation_videoqa_summary_df.to_csv(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV,
    index=False,
)

if not Path(REPRESENTATION_VIDEOQA_SUMMARY_CSV).exists():
    raise FileNotFoundError(
        f"Failed to create summary file: {REPRESENTATION_VIDEOQA_SUMMARY_CSV}"
    )

# ------------------------------------------------------------
# Verify saved local summary can be reloaded
# ------------------------------------------------------------

saved_summary_check_df = pd.read_csv(REPRESENTATION_VIDEOQA_SUMMARY_CSV)

if len(saved_summary_check_df) != len(representation_videoqa_summary_df):
    raise ValueError(
        "Saved summary file row count does not match "
        "the in-memory summary report."
    )

# ------------------------------------------------------------
# Promote summary report to Google Drive
# ------------------------------------------------------------

REPRESENTATION_VIDEOQA_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    REPRESENTATION_VIDEOQA_SUMMARY_CSV,
    REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV,
)

if not Path(REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV).exists():
    raise FileNotFoundError(
        f"Failed to promote summary file: "
        f"{REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV}"
    )

# ------------------------------------------------------------
# Display summary report
# ------------------------------------------------------------

print("Representation-based VideoQA summary report saved.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Prediction method  : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Choice accuracy    : {choice_accuracy:.3f}")
print(f"Local summary file : {REPRESENTATION_VIDEOQA_SUMMARY_CSV}")
print(f"Drive summary file : {REPRESENTATION_VIDEOQA_SUMMARY_DRIVE_CSV}")

display(representation_videoqa_summary_df)



### 🔷 Step 12 — Display Sample Predictions

* Randomly select representative prediction records from the representation-based VideoQA results.
* Display evaluation questions, ground-truth answers, predicted answers, and prediction scores.
* Show the prediction method and representation sources used to generate each prediction.
* Support qualitative assessment of representation-based multiple-choice prediction performance.
* Provide representative examples for experiment verification and debugging.

In [ ]:
# ============================================================
# Step 12: Display Sample Predictions
# ============================================================

import pandas as pd

print("Displaying sample representation-based VideoQA predictions...")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_step12_objects = [
    "representation_prediction_df",
    "random_seed",
    "dataset_mode_label",
    "evaluation_split",
    "REPRESENTATION_VIDEOQA_METHOD",
    "VIDEO_REPRESENTATION_SOURCE",
    "TEXT_REPRESENTATION_SOURCE",
]

missing_step12_objects = [
    name for name in required_step12_objects
    if name not in globals()
]

if missing_step12_objects:
    raise NameError(
        "Missing required Step 12 objects: "
        + ", ".join(missing_step12_objects)
    )

if representation_prediction_df.empty:
    raise RuntimeError(
        "representation_prediction_df is empty."
    )

# ------------------------------------------------------------
# Select sample predictions
# ------------------------------------------------------------

sample_count = min(
    10,
    len(representation_prediction_df),
)

sample_prediction_df = (
    representation_prediction_df
    .sample(
        n=sample_count,
        random_state=random_seed,
    )
    .sort_values(
        [
            "choice_correct",
            "video",
            "question",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validate display columns
# ------------------------------------------------------------

display_columns = [
    "qa_record_id",
    "video",
    "question",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "ground_truth",
    "prediction",
    "prediction_score",
    "prediction_method",
    "video_representation_source",
    "text_representation_source",
    "text_embedding_dimension",
    "video_embedding_dimension",
    "fusion_embedding_dimension",
]

missing_display_columns = [
    col for col in display_columns
    if col not in sample_prediction_df.columns
]

if missing_display_columns:
    raise ValueError(
        "Sample prediction display is missing required columns: "
        + ", ".join(missing_display_columns)
    )

# ------------------------------------------------------------
# Display sample predictions
# ------------------------------------------------------------

print(f"Displaying {sample_count} sample prediction records...")
print(f"Dataset mode                 : {dataset_mode_label}")
print(f"Evaluation split             : {evaluation_split}")
print(f"Prediction method            : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Video representation source  : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text representation source   : {TEXT_REPRESENTATION_SOURCE}")
print(f"Text embedding dimension     : {sample_prediction_df['text_embedding_dimension'].iloc[0]}")
print(f"Video embedding dimension    : {sample_prediction_df['video_embedding_dimension'].iloc[0]}")
print(f"Fusion embedding dimension   : {sample_prediction_df['fusion_embedding_dimension'].iloc[0]}")
print(
    f"Sample accuracy              : "
    f"{sample_prediction_df['choice_correct'].mean():.3f}"
)

display(
    sample_prediction_df[
        display_columns
    ]
)



### 🔷 Step 13 — Notebook Summary

* Summarize the completed representation-based VideoQA experiment.
* Report the experiment configuration, representation sources, Fusion MLP classifier, evaluation dataset size, and prediction accuracy.
* Display the locations of the generated prediction, validation, and summary artifacts.
* Confirm that all required experiment artifacts are ready for downstream evaluation in Notebook 08.
* Verify successful completion of the representation-based VideoQA pipeline.




In [ ]:
# ============================================================
# Step 13: Notebook Summary
# ============================================================

print("Notebook 07 complete.")
print("=" * 60)

print("\nRepresentation-Based VideoQA — Configuration")
print("-" * 60)
print(f"Dataset mode                 : {dataset_mode_label}")
print(f"Evaluation split             : {evaluation_split}")
print(f"Answer mode                  : {answer_mode}")
print(f"Prediction method            : {REPRESENTATION_VIDEOQA_METHOD}")
print(f"Video representation source  : {VIDEO_REPRESENTATION_SOURCE}")
print(f"Text representation source   : {TEXT_REPRESENTATION_SOURCE}")

print("\nInput Dataset Summary")
print("-" * 60)
print(f"QA input records             : {len(qa_input_df):,}")
print(f"Candidate answer rows        : {len(scored_candidate_df):,}")
print(f"Unique videos                : {representation_prediction_df['video'].nunique():,}")
print(f"Answer choices per question  : {NUM_MULTIPLE_CHOICE_ANSWERS}")

print("\nRepresentation Summary")
print("-" * 60)
print(f"Text embedding dimensions    : {text_embedding_dimension:,}")
print(f"Video embedding dimensions   : {video_embedding_dimension:,}")
print(f"Fusion embedding dimensions  : {FUSION_EMBEDDING_DIM:,}")
print(f"Text representation records  : {len(filtered_clip_text_df):,}")
print(f"Video representation records : {len(filtered_video_representation_df):,}")

print("\nFusion MLP Summary")
print("-" * 60)
print(f"Training epochs              : {FUSION_EPOCHS}")
print(f"Final training loss          : {fusion_training_history_df['loss'].iloc[-1]:.4f}")

print("\nPrediction Results")
print("-" * 60)
print(f"Prediction records           : {len(representation_prediction_df):,}")
print(f"Correct predictions          : {int(representation_prediction_df['choice_correct'].sum()):,}")
print(f"Choice accuracy              : {representation_choice_accuracy:.3f}")

print("\nOutput Artifacts")
print("-" * 60)
print(f"Prediction file              : {REPRESENTATION_VIDEOQA_PREDICTIONS_CSV}")
print(f"Validation file              : {REPRESENTATION_VIDEOQA_VALIDATION_CSV}")
print(f"Summary file                 : {REPRESENTATION_VIDEOQA_SUMMARY_CSV}")
print(f"Output directory             : {REPRESENTATION_VIDEOQA_LOCAL_DIR}")

print("\nNotebook 07 generated:")
print("- Fusion MLP representation-based VideoQA prediction dataset")
print("- Prediction validation report")
print("- Representation-based VideoQA summary report")
print("- Sample prediction records")

print("\nNotebook 07 outputs are ready for downstream evaluation.")

